**TODO:明天开始从头写，一点点替代PyTorch**

In [3]:
import numpy as np


class ReLU:
    def __init__(self):
        self.Z_l = None

    def forward(self, Z_l):
        self.Z_l = Z_l
        return np.maximum(0, Z_l)

    def backward(self, dL_da_l):
        return dL_da_l * (self.Z_l > 0)


class Linear:
    def __init__(self, in_dim, out_dim, lr=0.001):
        self.W_l = np.random.randn(in_dim, out_dim) * np.sqrt(2.0 / in_dim)
        self.b_l = np.zeros(out_dim)
        self.lr = lr
        self.a_l_1 = None
        self.dW = None
        self.db = None

    def forward(self, a_l_1):
        self.a_l_1 = a_l_1
        return self.a_l_1 @ self.W_l + self.b_l

    def backward(self, dL_dZ_l):
        self.dW = self.a_l_1.T @ dL_dZ_l
        self.db = np.sum(dL_dZ_l, axis=0)
        return dL_dZ_l @ self.W_l.T

    def step(self):
        self.W_l -= self.lr * self.dW
        self.b_l -= self.lr * self.db


class CrossEntropyLoss:
    def __init__(self):
        self.a_softmax = None
        self.t_onehot = None

    def forward(self, A, t):
        A_stable = A - np.max(A, axis=1, keepdims=True)
        A_exp = np.exp(A_stable)
        self.a_softmax = A_exp / np.sum(A_exp, axis=1, keepdims=True)

        x_shape = A.shape[0]
        if t.ndim == 1:
            t_onehot = np.zeros_like(self.a_softmax)
            t_onehot[np.arange(x_shape), t] = 1
            self.t_onehot = t_onehot
        else:
            self.t_onehot = t

        in_loss = -np.sum(self.t_onehot * np.log(self.a_softmax + 1e-12)) / x_shape
        return in_loss

    def backward(self):
        x_shape = self.t_onehot.shape[0]
        return (self.a_softmax - self.t_onehot) / x_shape


In [4]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST


train_data = MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(batch_size=64, shuffle=True, num_workers=0, dataset=train_data)

layers = [
    Linear(784, 128, lr=0.06),
    ReLU(),
    Linear(128, 64, lr=0.06),
    ReLU(),
    Linear(64, 10),
]
loss_fn = CrossEntropyLoss()

for epoch in range(5):
    epoch_loss = 0
    correct = 0
    total = 0

    for img, label in train_loader:
        element = img.view(-1, 784).numpy()
        target = label.numpy()

        for layer in layers:
            element = layer.forward(element)

        loss = loss_fn.forward(element, target)
        diff = loss_fn.backward()
        for layer in reversed(layers):
            diff = layer.backward(diff)

        for layer in layers:
            if isinstance(layer, Linear):
                layer.step()

        epoch_loss += loss
        correct += (element.argmax(1) == target).sum()
        total += len(target)

    print(f"Epoch={epoch+1}, Loss={epoch_loss/len(train_loader):.4f}, Accuracy={correct/total:.4f}")


Epoch=1, Loss=0.4164, Accuracy=0.8858
Epoch=2, Loss=0.2184, Accuracy=0.9380
Epoch=3, Loss=0.1716, Accuracy=0.9512
Epoch=4, Loss=0.1427, Accuracy=0.9601
Epoch=5, Loss=0.1220, Accuracy=0.9655
